In [8]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
import sys
import os

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.team_info import nameDict
from src.utils.nbaPlayerLogs import NBAGameLogs

pd.set_option('display.max_columns', None)

## Fetches Player Gamelogs

In [2]:
# nba = FetchPlayersStats()
# data = nba.getCompleteStats(
#     season='2025-26', 
#     season_type='Regular Season', 
#     sleep_time=2, 
#     max_workers=5,
#     batch_limit=100,
#     complete_cache_file='data/raw/season_stats/S26.csv',
#     include_playbyplay=False
# )
# data.tail()

In [2]:
# df = NBAGameLogs(season='2024-25', season_type='Regular Season').fetch(skip_start_positions=True).build().get_df()
# df.head()

# Session by session — run this repeatedly until all games are done
logs = NBAGameLogs(season='2025-26', season_type='Playoffs')
logs.fetch(batch_size=50, start_position_delay=2.5, start_position_workers=5, checkpoint_path='tracking_checkpoint.csv')

Fetching data for 2025-26 Playoffs...
✓ Player base
✓ Player advanced
✓ Team base
✓ Team advanced
✓ Checkpoint loaded — 7669/30 games already done, 0 remaining
✓ All games already fetched from checkpoint
✓ START_POSITION ready (201190 rows)


In [ ]:
df = logs.build().get_df()
def normalize_player_names(frame, col='PLAYER_NAME', mapping=nameDict):
    """Rewrite player names to the canonical form used as keys in ``mapping``.

    ``mapping`` is {desired_name: current_name_in_data}, so we invert it and
    use ``Series.replace`` to swap any matching value back to the key form.
    """
    inv = {v: k for k, v in mapping.items()}
    before = frame[col].isin(inv).sum()
    frame[col] = frame[col].replace(inv)
    print(f"Renamed {before} rows in '{col}'")
    return frame


df = normalize_player_names(df)
df.head()

In [4]:
pos = pd.read_csv('/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/raw/player_positions/nba_2026_players.csv').rename(
    columns={'name_s26': 'PLAYER_NAME', 'pos': 'POS', 'age': 'AGE'}
)
starting_positions = pd.read_csv('tracking_checkpoint.csv')
df = df.merge(pos, on='PLAYER_NAME', how='left')
df.head()

✓ Built — shape: (671, 173)


,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,START_POSITION,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,name,POS,AGE
0,2025-26,1641705,Victor Wembanyama,Victor,1610612759,SAS,San Antonio Spurs,0042500154,2026-04-26T00:00:00,SAS @ POR,W,34.316667,9,17,0.529,1,4,0.250,8,8,1.000,1,11,12,3,4,4,7,0,3,9,27,28,74.9,1,0,65.0,1,34:19,1,122.7,120.5,120.5,82.1,83.3,83.3,40.6,37.2,37.2,0.125,0.75,10.7,0.033,0.289,0.176,14.3,14.5,0.559,0.658,0.316,0.320,101.24,101.41,84.51,101.41,0.287,73,9.0,17.0,C,4.03,2.48,2,17,19,60,0,0,32,5,7,0.714,4,10,0.4,3,8,0.375,43,87,0.494,14,33,0.424,14,17,0.824,9,31,40,26,13.0,12,10,5,21,20,114,21.0,115.8,116.3,92.0,93.0,23.8,23.3,0.605,2.00,19.4,0.217,0.725,0.485,0.133,0.575,0.603,99.8,99.0,82.5,98,0.646,1610612757,POR,Portland Trail Blazers,32,80,0.400,10,31,0.323,19,23,0.826,7,32,39,14,18.0,6,5,10,20,21,93,-21.0,92.0,93.0,115.8,116.3,-23.8,-23.3,0.438,0.78,10.9,0.275,0.783,0.515,0.180,0.463,0.516,99.8,99.0,82.5,100,0.354,Victor Wembanyama,C,22.0
1,2025-26,1628369,Jayson Tatum,Jayson,1610612738,BOS,Boston Celtics,0042500114,2026-04-26T00:00:00,BOS @ PHI,W,34.883333,8,16,0.500,5,10,0.500,9,9,1.000,0,7,7,11,5,0,0,2,3,6,30,25,49.9,1,0,53.0,1,34:53,1,136.9,146.3,146.3,106.6,109.0,109.0,30.3,37.3,37.3,0.478,2.20,31.4,0.000,0.200,0.100,14.3,13.9,0.656,0.752,0.289,0.299,96.35,92.19,76.83,92.19,0.218,67,8.0,16.0,F,3.59,2.24,3,11,13,88,0,2,61,5,9,0.556,3,7,0.429,0,0,0.0,42,87,0.483,24,53,0.453,20,28,0.714,14,37,51,28,13.0,5,5,7,21,21,128,32.0,130.2,139.1,102.6,102.1,27.6,37.0,0.667,2.15,20.1,0.396,0.796,0.598,0.141,0.621,0.644,95.9,93.0,77.5,92,0.609,1610612755,PHI,Philadelphia 76ers,33,80,0.413,9,30,0.300,21,24,0.875,6,24,30,25,9.0,6,7,5,21,21,96,-32.0,102.6,102.1,130.2,139.1,-27.6,-37.0,0.758,2.78,19.4,0.204,0.604,0.402,0.096,0.469,0.530,95.9,93.0,77.5,94,0.391,Jayson Tatum,PF,27.0
2,2025-26,1628368,De'Aaron Fox,De'Aaron,1610612759,SAS,San Antonio Spurs,0042500154,2026-04-26T00:00:00,SAS @ POR,W,38.750000,11,17,0.647,4,8,0.500,2,4,0.500,1,5,6,7,3,1,2,1,1,3,28,21,51.7,0,0,51.0,1,38:45,1,119.1,117.1,117.1,91.6,92.6,92.6,27.6,24.5,24.5,0.280,2.33,24.1,0.030,0.122,0.081,10.3,10.4,0.765,0.746,0.250,0.254,100.66,100.95,84.13,100.95,0.218,82,11.0,17.0,G,4.09,2.84,2,7,9,75,0,0,49,5,6,0.833,6,11,0.545,1,2,0.5,43,87,0.494,14,33,0.424,14,17,0.824,9,31,40,26,13.0,12,10,5,21,20,114,21.0,115.8,116.3,92.0,93.0,23.8,23.3,0.605,2.00,19.4,0.217,0.725,0.485,0.133,0.575,0.6

In [6]:
df.to_csv('/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/raw/playoff_stats/P26.csv')

In [2]:
"""
BettingPros NBA Prop Bets Scraper
Fetches each market separately to get all props.
"""

import requests
import json
import csv
from datetime import date

BASE_URL = "https://api.bettingpros.com/v3/props"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/121.0.0.0 Safari/537.36",
    "Accept": "application/json",
    "Accept-Language": "en-US,en;q=0.9",
    "Origin": "https://www.bettingpros.com",
    "Referer": "https://www.bettingpros.com/",
}

MARKET_NAMES = {
    156: "Points",
    151: "Assists",
    157: "Rebounds",
    335: "Pts+Ast",
    336: "Pts+Reb",
    337: "Reb+Ast",
    338: "Pts+Reb+Ast",
    152: "Steals",
    160: "Blocks",
    162: "3-Pointers Made",
}


def fetch_market(target_date: str, market_id: int, limit: int = 100, offset: int = 0) -> dict:
    params = {
        "limit": limit,
        "offset": offset,
        "sport": "NBA",
        "market_id": market_id,
        "date": target_date,
        "include_selections": "false",
        "include_filter_graphs": "false",
        "data_points": 8,
        "min_odds": -1000,
        "max_odds": 1000,
        "ev_threshold_min": -0.4,
        "ev_threshold_max": 0.4,
    }
    resp = requests.get(BASE_URL, headers=HEADERS, params=params, timeout=15)
    resp.raise_for_status()
    return resp.json()


def parse_props(data: dict) -> list[dict]:
    rows = []
    for prop in data.get("props", []):
        proj = prop.get("projection") or {}
        rows.append({
            "player": prop.get("participant", {}).get("name", "Unknown"),
            "prop":   MARKET_NAMES.get(prop.get("market_id"), prop.get("market_id")),
            "line":   prop.get("over", {}).get("line"),
            "proj":   proj.get("value"),
            "side":   proj.get("recommended_side"),
            "diff":   proj.get("diff"),
        })
    return rows


def scrape_all(target_date: str) -> list[dict]:
    all_rows = []

    print(f"Fetching props for {target_date}...")
    for market_id, market_name in MARKET_NAMES.items():
        seen   = set()
        offset = 0

        while True:
            data = fetch_market(target_date, market_id, limit=500, offset=offset)
            rows = parse_props(data)

            if not rows:
                break

            new_rows = []
            for r in rows:
                key = (r["player"], r["prop"], r["line"])
                if key not in seen:
                    seen.add(key)
                    new_rows.append(r)

            if not new_rows:
                break

            all_rows.extend(new_rows)
            offset += 100

        print(f"  {market_name:<20} → {len([r for r in all_rows if r['prop'] == market_name])} props")

    print(f"\n  Total: {len(all_rows)} props")
    return all_rows


def save_csv(rows: list[dict], filename: str):
    if not rows:
        print("No data to save.")
        return
    fieldnames = list(rows[0].keys())
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f"Saved {len(rows)} rows → {filename}")


def save_json(rows: list[dict], filename: str):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2)
    print(f"Saved {len(rows)} rows → {filename}")


from datetime import date, timedelta

START_DATE    = date(2025, 4, 1)
END_DATE      = date(2025, 4, 15)
OUTPUT_FORMAT = "csv"
OUTPUT_NAME   = "nba_props"

current = START_DATE
while current <= END_DATE:
    target = str(current)
    print(f"\n{'='*50}\nProcessing {target}\n{'='*50}")
    
    rows = scrape_all(target)
    
    if rows:
        if OUTPUT_FORMAT in ("csv", "both"):
            save_csv(rows, f"historical_odds/{target}.csv")
        if OUTPUT_FORMAT in ("json", "both"):
            save_json(rows, f"historical_odds/{target}.json")
    else:
        print(f"  No props found for {target}, skipping.")
    
    current += timedelta(days=1)

print("\nDone!")


Processing 2025-04-01
Fetching props for 2025-04-01...
  Points               → 76 props
  Assists              → 81 props
  Rebounds             → 78 props
  Pts+Ast              → 75 props
  Pts+Reb              → 73 props
  Reb+Ast              → 73 props
  Pts+Reb+Ast          → 72 props
  Steals               → 76 props
  Blocks               → 83 props
  3-Pointers Made      → 76 props

  Total: 763 props


FileNotFoundError: [Errno 2] No such file or directory: 'historical_odds/2025-04-01.csv'